In [4]:
import pandas as pd
import os

def remove_spaces(raw_keywords):
    """
    한 행의 키워드 데이터를 받아 쉼표 기준으로 분리한 뒤,
    각 키워드 내부의 공백(띄어쓰기)을 모두 제거하여 덩어리 키워드로 만듭니다.
    """
    # 값이 비어있을 경우(NaN) 그대로 반환
    if pd.isna(raw_keywords):
        return raw_keywords
    
    final_words = []
    # 1. 쉼표(,)로 분리
    comma_split = str(raw_keywords).split(',')
    
    for item in comma_split:
        # 2. 키워드 내의 모든 공백(띄어쓰기) 제거
        # 예: '인테리어 소품' -> '인테리어소품'
        word_no_space = item.replace(" ", "")
        
        # 공백만 있던 문자열이 아니었다면 리스트에 추가
        if word_no_space: 
            final_words.append(word_no_space)
            
    # 분리된 단어들을 다시 쉼표(,)로 연결하여 문자열로 반환
    return ", ".join(final_words)

def main():
    # 파일 경로 설정
    input_file = '/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords.csv'
    output_file = '/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv'
    
    if not os.path.exists(input_file):
        print(f"파일을 찾을 수 없습니다: {input_file}")
        return

    # CSV 파일 읽기
    try:
        df = pd.read_csv(input_file)
    except Exception as e:
        print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
        return

    if 'keywords' not in df.columns:
        print("CSV 파일에 'keywords' 컬럼이 없습니다.")
        return

    # === 핵심 변경 부분: 띄어쓰기 제거 함수 적용 ===
    df['keywords_v2'] = df['keywords'].apply(remove_spaces)

    # 결과 확인 출력
    print("데이터 변환 완료! 변환된 샘플 5개를 확인합니다:")
    print(df[['keywords', 'keywords_v2']].head())

    # 새로운 CSV 파일로 저장
    try:
        df.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"\n새로운 파일이 성공적으로 저장되었습니다:\n{output_file}")
    except Exception as e:
        print(f"파일을 저장하는 중 오류가 발생했습니다: {e}")

if __name__ == "__main__":
    main()


데이터 변환 완료! 변환된 샘플 5개를 확인합니다:
                                            keywords  \
0  럭셔리, 스테이, 리조트, 아만 다리, 호시노야 가루이자와, 스테이폴리오, 인플루언...   
1    롯데리아, 나폴리, 감자연구소, 디저트, 쥐포튀김, 떼리앙, 캐릭터, 띠부씰, 콜라보   
2          국밥, 뉴웨이브, MZ세대, 힙함, 감성, 이색 식재료, 플레이팅, 한 끼   
3      성수, 합정, 홍대, 을지로, 데이트, 카페, 핫플레이스, LP, 퇴근후, 아지트   
4         오이, 소금빵, 디저트, 샌드위치, 바게트, 고리, 슬로우타운, 베이커리호프   

                                         keywords_v2  
0  럭셔리, 스테이, 리조트, 아만다리, 호시노야가루이자와, 스테이폴리오, 인플루언서,...  
1    롯데리아, 나폴리, 감자연구소, 디저트, 쥐포튀김, 떼리앙, 캐릭터, 띠부씰, 콜라보  
2            국밥, 뉴웨이브, MZ세대, 힙함, 감성, 이색식재료, 플레이팅, 한끼  
3      성수, 합정, 홍대, 을지로, 데이트, 카페, 핫플레이스, LP, 퇴근후, 아지트  
4         오이, 소금빵, 디저트, 샌드위치, 바게트, 고리, 슬로우타운, 베이커리호프  

새로운 파일이 성공적으로 저장되었습니다:
/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv


In [9]:
# 1. 데이터가 잘 들어왔는지 먼저 확인

file_path = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
column_name = "keywords_v2"


print(f"데이터 로드 중: {file_path}")
df = pd.read_csv(file_path)

print(f"데이터 행 개수: {len(df)}")
print(f"첫 번째 행 데이터 타입: {type(df['keywords'].iloc[0])}")
print("첫 번째 행 실제값:", repr(df['keywords_v2'].iloc[0]))


데이터 로드 중: /Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv
데이터 행 개수: 1037
첫 번째 행 데이터 타입: <class 'str'>
첫 번째 행 실제값: '럭셔리, 스테이, 리조트, 아만다리, 호시노야가루이자와, 스테이폴리오, 인플루언서, 여행, 숙박'


In [ ]:
import pandas as pd
import networkx as nx
from itertools import combinations
import ast
import os

def safe_eval(x):
    """
    문자열 형태의 리스트(['A', 'B'])를 실제 파이썬 리스트 객체로 안전하게 변환합니다.
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        # 변환 실패 시 공백 제거 후 리스트화 시도하거나 그대로 반환
        return x

def main():
    # 1. 데이터 로드 및 전처리
    file_path = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
    column_name = "keywords_v2"
    
    if not os.path.exists(file_path):
        print(f"파일을 찾을 수 없습니다: {file_path}")
        return

    print(f"데이터 로드 중: {file_path}")
    df = pd.read_csv(file_path)
    
    if column_name not in df.columns:
        print(f"오류: '{column_name}' 컬럼이 데이터프레임에 존재하지 않습니다.")
        return

    # 데이터 타입 변환 (문자열 -> 리스트)
    df[column_name] = df[column_name].apply(safe_eval)

    # 2. 네트워크 모델링 (NetworkX)
    print("동시 출현 네트워크(Co-occurrence Network) 생성 중...")
    G = nx.Graph()

    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue
            
        # 한 게시물 내 단어들로 가능한 모든 2개 조합 추출
        # 정렬(sorted)을 통해 (A, B)와 (B, A)가 동일하게 취급되도록 함
        for u, v in combinations(keywords, 2):
            if G.has_edge(u, v):
                # 이미 존재하면 가중치(Weight)를 1 누적
                G[u][v]['weight'] += 1
            else:
                # 처음 나타나면 엣지 생성 및 가중치 1 부여
                G.add_edge(u, v, weight=1)

    # 3. 결과 검증 및 통계 출력
    print("-" * 50)
    print(f"네트워크 구축 완료!")
    print(f"전체 노드(키워드) 수: {G.number_of_nodes()}")
    print(f"전체 엣지(연결) 수: {G.number_of_edges()}")
    print("-" * 50)

    # 가중치(동시 출현 빈도) 기준 상위 10개 단어 쌍 출력
    sorted_edges = sorted(G.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)
    
    print("상위 10개 단어 쌍 (동시 출현 빈도 기준):")
    for u, v, data in sorted_edges[:10]:
        print(f"[{u}] - [{v}] : {data['weight']}회")
    
        # 네트워크 기본 통계
    print(f"평균 연결 중심성: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")
    print(f"네트워크 밀도: {nx.density(G):.4f}")

if __name__ == "__main__":
    main()


데이터 로드 중: /Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv
동시 출현 네트워크(Co-occurrence Network) 생성 중...
--------------------------------------------------
네트워크 구축 완료!
전체 노드(키워드) 수: 0
전체 엣지(연결) 수: 0
--------------------------------------------------
상위 10개 단어 쌍 (동시 출현 빈도 기준):


ZeroDivisionError: division by zero

In [10]:
import pandas as pd
import networkx as nx
from itertools import combinations
import ast
import os


def safe_eval(x):
    """
    다양한 형태의 키워드 데이터를 파이썬 리스트 객체로 안전하게 변환합니다.
    - 형태 A: "['디저트', '당충전']"  → ast.literal_eval 처리
    - 형태 B: "디저트, 당충전, 재충전" → split(',') 처리  ← 현재 데이터 형식
    - 형태 C: "[디저트, 당충전]"       → 대괄호 제거 후 split(',') 처리
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        # 형태 A 시도
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            # 형태 C: 대괄호 제거 후 형태 B와 동일하게 처리
            cleaned = x.strip('[]')
            # 형태 B: 쉼표 기준으로 분리하고 공백 제거
            return [kw.strip() for kw in cleaned.split(',') if kw.strip()]
        return []


def main():
    # 1. 데이터 로드 및 전처리
    file_path = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
    column_name = "keywords_v2"

    if not os.path.exists(file_path):
        print(f"파일을 찾을 수 없습니다: {file_path}")
        return

    print(f"데이터 로드 중: {file_path}")
    df = pd.read_csv(file_path)
    print(f"데이터 행 개수: {len(df)}")

    if column_name not in df.columns:
        print(f"오류: '{column_name}' 컬럼이 데이터프레임에 존재하지 않습니다.")
        return

    # 데이터 타입 변환 (문자열 -> 리스트)
    df[column_name] = df[column_name].apply(safe_eval)

    # 변환 결과 샘플 확인
    print("\n[변환 결과 샘플]")
    for i, val in enumerate(df[column_name].head(3)):
        print(f"  [{i}] {val}")

    # 2. 고빈도 불용어 처리 (전체 게시물의 30% 이상 등장 키워드 제거)
    from collections import Counter

    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):  # 게시물당 1회만 카운트
            kw_doc_freq[kw] += 1

    threshold = total_docs * 0.30
    stopwords = {kw for kw, freq in kw_doc_freq.items() if freq >= threshold}

    if stopwords:
        print(f"\n[고빈도 불용어 제거] {len(stopwords)}개: {stopwords}")
    else:
        print("\n[고빈도 불용어] 해당 없음 (임계값: 전체 게시물의 30%)")

    # 3. 네트워크 모델링 (NetworkX)
    print("\n동시 출현 네트워크(Co-occurrence Network) 생성 중...")
    G = nx.Graph()

    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue

        # 불용어 제거 후 유효 키워드만 추출
        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue

        # 한 게시물 내 단어들로 가능한 모든 2개 조합 추출
        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

    # 4. 결과 검증 및 통계 출력
    print("-" * 50)
    print(f"네트워크 구축 완료!")
    print(f"전체 노드(키워드) 수: {G.number_of_nodes()}")
    print(f"전체 엣지(연결) 수:   {G.number_of_edges()}")
    print("-" * 50)

    if G.number_of_nodes() == 0:
        print("네트워크가 비어 있습니다. 데이터 형식을 다시 확인하세요.")
        return

    # 가중치(동시 출현 빈도) 기준 상위 10개 단어 쌍 출력
    sorted_edges = sorted(G.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)
    print("\n상위 10개 단어 쌍 (동시 출현 빈도 기준):")
    for u, v, data in sorted_edges[:10]:
        print(f"  [{u}] - [{v}] : {data['weight']}회")

    # 네트워크 기본 통계
    print("\n[네트워크 기본 통계]")
    avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
    print(f"  평균 연결 중심성 (Avg Degree): {avg_degree:.2f}")
    print(f"  네트워크 밀도 (Density):       {nx.density(G):.4f}")
    # 밀도 해석 안내
    density = nx.density(G)
    if density < 0.01:
        print("  → 희소(Sparse) 네트워크 — 정상 범위")
    elif density < 0.1:
        print("  → 보통 밀도 — 정상 범위")
    else:
        print("  → 고밀도 — 불용어 추가 처리 검토 권장")


if __name__ == "__main__":
    main()

데이터 로드 중: /Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv
데이터 행 개수: 1037

[변환 결과 샘플]
  [0] ['럭셔리', '스테이', '리조트', '아만다리', '호시노야가루이자와', '스테이폴리오', '인플루언서', '여행', '숙박']
  [1] ['롯데리아', '나폴리', '감자연구소', '디저트', '쥐포튀김', '떼리앙', '캐릭터', '띠부씰', '콜라보']
  [2] ['국밥', '뉴웨이브', 'MZ세대', '힙함', '감성', '이색식재료', '플레이팅', '한끼']

[고빈도 불용어] 해당 없음 (임계값: 전체 게시물의 30%)

동시 출현 네트워크(Co-occurrence Network) 생성 중...
--------------------------------------------------
네트워크 구축 완료!
전체 노드(키워드) 수: 3982
전체 엣지(연결) 수:   30508
--------------------------------------------------

상위 10개 단어 쌍 (동시 출현 빈도 기준):
  [카페] - [맛집] : 70회
  [디저트] - [카페] : 59회
  [카페] - [브런치] : 43회
  [여행] - [카페] : 41회
  [카페] - [커피] : 40회
  [감성] - [카페] : 31회
  [카페] - [F&B] : 28회
  [카페] - [빈티지] : 28회
  [카페] - [힐링] : 27회
  [디저트] - [커피] : 26회

[네트워크 기본 통계]
  평균 연결 중심성 (Avg Degree): 15.32
  네트워크 밀도 (Density):       0.0038
  → 희소(Sparse) 네트워크 — 정상 범위
